## Tarea 5: TP Máquina de Turing Universal

*1 - ¿Por qué se dice que una MTU es capaz de simular cualquier Máquina de Turing?*

Se dice que una MTU es capaz de simular cualquier MT porque es capaz de recibir cualquier MT Codificada (M) junto con una cadena de entrada (w) y hacer toda la simulacion de como actuaria la entrada w sobre la MT M.

- U(< M, w >) = M(w)
 - U = Maquina Universal de Turing
 - < M, w > = Entrada de MTU
 - M = MT Codificada
 - w = Entrada de la MT
 - M(w) = Resultado de usar w como entrada en M 

*2 - Suponer que se tiene una MT M que acepta todas las cadenas que terminan en 01. Indicar qué debería hacer una MTU con las siguientes entradas. Explicar en cada caso si la MTU acepta o rechaza y por qué*

- U(⟨M⟩,1101) -> MTU acepta la entrada porque M acepta cualquier w terminada en 01.
- U(⟨M⟩,100) -> MTU rechaza la entrada porque M solo acepta cualquier w terminada en 01.
- U(⟨M⟩,01) -> MTU acepta la entrada porque M acepta cualquier w terminada en 01.
- U(⟨M⟩,111) -> MTU rechaza la entrada porque M solo acepta cualquier w terminada en 01.

*3 - Dada la siguiente MT M:*

| Q    | 0            | 1            |
| ---- | ------------ | ------------ |
| q0   | (q1,1,R)     | (q1,0,R)     |
| q1   | (qf,0,R)     | (qf,1,R)     |
| qf   | -            | -            |

*3A - Explicar que hace M*

*M* es una MT que **invierte el primer bit** de una entrada *w* valida.

- M = < Q, Σ, Γ, δ, q0, □, F >
- _Σ = {0, 1}_
- _Γ = {0, 1,  □}_
- _□ ∈ Γ_ 
- _Q = {q0, q1, qf}_
- _q0 ∈ Q_ 
- _F = { qf}_ 
- _δ: Q x Γ -> Q x Γ x {L, R, S}_

*3B - Explicar qué información debería recibir una MTU para poder simular M*

MTU deberia recibir la MT M codificada junto con una entrada wp ara poder ser capaz de simularla. Para ello necesita
- Codificacion de sus simbolos (0 y 1)
- Codificacion de sus estados (q0, q1, qf)
- Codificacion de los movimientos (L, R, S)
- Configuracion de la cinta (Estado inicial y Posicion del cabezal)
- Entrada w

*3C - Codificar la cintar de MTU sabiendo que configuración de la cinta de MT M es 1 q0 0 1 1*

*Codificacion:*
- Simbolos: 0 = 0; 1 = 1;
- Estados: q0 = 00; q1 = 01; qf= 10
- Movimientos: S = 00; L = 10; R = 01 
- Entrada w: 1q0001

*Codificacion:*
- Cinta y Cabezal: 1*001
- Configuracion actual: $000
- Transiciones:
 - #00001101 
 - #00101001
 - #01010001
 - #01110101
- Final: 1*001$000#00001101#00101001#01010001#01110101

## Practica

*1 - Codificación de una máquina simple*

Definir una máquina que recibe un número binario (cadena no vacía de 0’s y 1’s) y devuelve el siguiente número binario (es decir, le suma 1) Codificar sus estados, símbolos y transiciones en forma numérica*

- *M = ⟨ Γ, Σ, □, Q, q0, F, δ ⟩*
- Σ = { 0, 1 }
- Γ  = { 0, 1, □ }
- □ ∈ Γ
- _Q = { q0, q1, qf}_
- _q0 ∈ Q (Estado inicial)_
- _F = { qf }_ 
- _δ = Q x Γ → Q x Γ { L, R, S }_

| Estado Actual | 0           | 1               | □ (Blanco) |
| ------------- | ----------  | ----------      | ---------- |
| **>q0**       | (q0, 0, R)  | (q0, 1, R)      | (q1, □, L) |
| **q1**   | (qf, 1, S) | (q1, 0, L) | (qf, 1, S)      |
| **\*qf**     | -           | -               | -                |


### Codificación
- Simbolos: 0 = 00; 1 = 01; □ = 10
- Movimientos: S = 00; L = 10; R = 01
- Estados: q0 = 00; q1 = 01; qf = 10
- Transiciones: 
 - 00_00_00_00_01: (q0, 0) -> (q0, 0 , R) -> #0000000001
 - 00_01_00_01_01: (q0, 1) -> (q0, 1 , R) -> #0001000101
 - 00_10_01_10_10: (q0, □) -> (q1, □ , L) -> #0010011010
 - 01_00_10_01_00: (q1, 0) -> (qf, 1 , S) -> #0100100100
 - 01_01_01_00_10: (q1, 1) -> (q1, 0 , L) -> #0101010010
 - 01_10_10_01_00: (q1, □) -> (qf, 1 , S) -> #0110100100
- Ejemplo: *0001$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100

*2 - Simulación básica*
 
Implementar en Python un programa que reciba:
- La codificación de una máquina M
- Una cadena de entrada w
El programa debe simular paso a paso la ejecución de M sobre w 

In [ ]:

#Simulador paso a paso de una Maquina de Turing codificada
BLANCO = '10' 

COD_MOVIMIENTO = {
    "L": '10',
    "R": '01',
    "S": '00',
}

def obtener_entrada(mt):
    """
    Retrona el contenido de la cinta y la posicion del cabezal
    """
    return mt.split('$', 1)[0]


def obtener_estado(mt, ancho_transicion, ancho_valor):
    """
    Retorna una tupla (estado, valor_leido) 
    """
    resto = mt.split('$', 1)[1]            
    segmento_estado = resto.split('#', 1)[0] 

    estado = segmento_estado[:ancho_transicion]
    valor_leido = segmento_estado[ancho_transicion: ancho_transicion + ancho_valor]

    return estado, valor_leido


def obtener_transiciones(mt, ancho_transicion, ancho_valor):
    """
    Devuelve la lista completa de transiciones codificadas
    cada una como un diccionario con sus 5 campos.
    """
    ancho_movimiento = 2
    bloques = mt.split('#')[1:]  
    largo_bloque = 2 * ancho_transicion + 2 * ancho_valor + ancho_movimiento

    transiciones = []
    for bloque in bloques:
        bloque = bloque[:largo_bloque]  

        i = 0
        estado_actual = bloque[i:i + ancho_transicion]; i += ancho_transicion
        leyendo       = bloque[i:i + ancho_valor];       i += ancho_valor
        estado_nuevo  = bloque[i:i + ancho_transicion];  i += ancho_transicion
        escribir      = bloque[i:i + ancho_valor];       i += ancho_valor
        mover         = bloque[i:i + ancho_movimiento]

        transiciones.append({
            "estado_actual": estado_actual,
            "leyendo": leyendo,
            "estado_nuevo": estado_nuevo,
            "escribir": escribir,
            "mover": mover,
        })
    return transiciones


def buscar_transicion_del_estado(transiciones, estado, valor):
    """
    Retorna la transicion que conindice con el estado y valor pasados como parametro 
    """
    for transicion in transiciones:
        if transicion["estado_actual"] == estado and transicion["leyendo"] == valor:
            return transicion
    return None


# Funciones sobre la cinta

def separar_cinta_y_cabezal(entrada, ancho_valor):
    """
    A partir de la cinta codificada con '*' (ej: '101*01')
    Devuelve (lista_de_simbolos, indice_cabezal).
    """
    pos_cabezal = entrada.index('*')
    valor_cinta = entrada[:pos_cabezal] + entrada[pos_cabezal + 1:]

    indice_cabezal = pos_cabezal // ancho_valor
    simbolos = [valor_cinta[i:i + ancho_valor] for i in range(0, len(valor_cinta), ancho_valor)]
    return simbolos, indice_cabezal


def simular_transicion(simbolos, indice_cabezal, transicion, ancho_valor):
    """
    Aplica UNA transicion sobre la cinta:
    Si el cabezal se sale de la cinta actual, se agrega una celda en blanco
    (simulando una cinta infinita).
    Devuelve (nuevos_simbolos, nuevo_indice_cabezal).
    """
    simbolos = list(simbolos)  # copiamos para no pisar la cinta anterior
    simbolos[indice_cabezal] = transicion["escribir"]

    if transicion["mover"] == COD_MOVIMIENTO["L"]:      
        nuevo_indice = indice_cabezal - 1
        if nuevo_indice < 0:
            simbolos.insert(0, BLANCO)
            nuevo_indice = 0
    elif transicion["mover"] == COD_MOVIMIENTO["R"]:                                 
        nuevo_indice = indice_cabezal + 1 
        if nuevo_indice >= len(simbolos):
            simbolos.append(BLANCO)
    else:
        nuevo_indice = indice_cabezal #S

    return simbolos, nuevo_indice


def obtener_codificacion(simbolos, indice_cabezal, estado_nuevo, transiciones):
    """
    Arma de nuevo el string completo 'mt' a partir de:
      - la cinta actual (como lista de simbolos) + posicion del cabezal
      - el nuevo estado
      - las transiciones (que son siempre las mismas, no cambian nunca)
    """
    ancho_valor = len(simbolos[0])
    cinta_bits = "".join(simbolos)
    pos_cabezal = indice_cabezal * ancho_valor
    cinta_con_cabezal = cinta_bits[:pos_cabezal] + "*" + cinta_bits[pos_cabezal:]

    valor_bajo_cabezal = simbolos[indice_cabezal]  # lo que "se lee" ahora, ya movido el cabezal
    segmento_estado = "$" + estado_nuevo + valor_bajo_cabezal

    segmento_transiciones = "".join(
        "#" + t["estado_actual"] + t["leyendo"] + t["estado_nuevo"] + t["escribir"] + t["mover"]
        for t in transiciones
    )

    return cinta_con_cabezal + segmento_estado + segmento_transiciones


# printeo

def mostrar_step(numero_step, mt, estado, valor, transicion):
    """Imprime UNA ejecucion de la MT simulada"""
    cinta = obtener_entrada(mt)

    print(f"#T{numero_step}: {mt}")
    print(f"- Cinta: {cinta}")
    print(f"- Estado actual: {estado}(*{valor})")

    if transicion is None:
        print("- No hay transicion aplicable: la maquina se DETIENE ")
    else:
        if transicion["mover"] == COD_MOVIMIENTO["L"]:
            direccion = "L"
        elif transicion["mover"] == COD_MOVIMIENTO["R"]:
            direccion = "R"
        else:
            direccion = "S"
        print(
            f"- Transicion: δ ({estado}, {valor}) = "
            f"({transicion['estado_nuevo']}, {transicion['escribir']}, {direccion})"
        )
    print()


# main

def simular_ejecucion(mt, config, steps=5, numero_step=1):
    """
    Simula 'steps' pasos de la maquina de Turing codificada en 'mt',
    imprimiendo cada configuracion (C1, C2, C3, ...) por consola.

    config debe tener las claves:
        "ANCHO_VALOR"      -> cuantos bits ocupa cada simbolo
        "ANCHO_TRANSICION" -> cuantos bits ocupa cada estado
    """
    ancho_valor = config["ANCHO_VALOR"]
    ancho_transicion = config["ANCHO_TRANSICION"]

    entrada = obtener_entrada(mt)
    estado, valor = obtener_estado(mt, ancho_transicion, ancho_valor)
    lista_transiciones = obtener_transiciones(mt, ancho_transicion, ancho_valor)


    transicion = buscar_transicion_del_estado(lista_transiciones, estado, valor)

    mostrar_step(numero_step, mt, estado, valor, transicion)

    if transicion is None or steps <= 1:
        return

    simbolos, indice_cabezal = separar_cinta_y_cabezal(entrada, ancho_valor)
    nuevos_simbolos, nuevo_indice = simular_transicion(simbolos, indice_cabezal, transicion, ancho_valor)

    nueva_codificacion = obtener_codificacion(
        nuevos_simbolos, nuevo_indice, transicion["estado_nuevo"], lista_transiciones
    )

    simular_ejecucion(nueva_codificacion, config, steps - 1, numero_step + 1)


#Programa principal
if __name__ == "__main__":

    print(" Simulador de Maquina de Turing codificada ")

    # Pedir datos de configuracion.
    mt_ingresada = input("Codificacion de la maquina M (con w ya incluido en la cinta): ").strip()
    ancho_valor = int(input("ANCHO_VALOR (bits por simbolo): ").strip())
    ancho_transicion = int(input("ANCHO_TRANSICION (bits por estado): ").strip())
    cantidad_pasos = int(input("Cantidad de pasos a simular: ").strip())
    configuracion = {
        "ANCHO_VALOR": ancho_valor,
        "ANCHO_TRANSICION": ancho_transicion,
    }
    print()
    simular_ejecucion(mt_ingresada, configuracion, steps=cantidad_pasos)

    """
    DATOS DE PRUEBA:
    MT = Sumar 1 bit al numero binario que esta en la cinta (ej: 001 -> 010)
    ### Codificación
    - Simbolos: 0 = 00; 1 = 01; □ = 10
    - Movimientos: 00 = S; 10 = L; 01 = R
    - Estados: q0 = 00; q1 = 01; qf = 10
    - Transiciones: 
    - 00_00_00_00_01: (q0, 0) -> (q0, 0 , R) -> #0000000001
    - 00_01_00_01_01: (q0, 1) -> (q0, 1 , R) -> #0001000101
    - 00_10_01_10_10: (q0, □) -> (q1, □ , L) -> #0010011010
    - 01_00_10_01_00: (q1, 0) -> (qf, 1 , S) -> #0100100100
    - 01_01_01_00_10: (q1, 1) -> (q1, 0 , L) -> #0101010010
    - 01_10_10_01_00: (q1, □) -> (qf, 1 , S) -> #0110100100
    - Ejemplo: *0001$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
    w = 01
    ANCHO_VALOR = 2
    ANCHO_ESTADO = 2
    ANCHO_MOVIMIENTO = 2
    MT_COD = *0001$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
    """

## Pruebas de Funcionamiento

Se uso la MT codificada arriba que sumaba 1 numero binario a una cadena w.

### Prueba N°1

Sumar 1 al numero binario 01: (01 + 1) = (10)

- M = *0001$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta = q001
- Cinta final esperada = qf1000
- Resultado final = *01001010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100

#### Salidas del código:

```
#T1: *0001$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *0001
- Estado actual: 00(*00)
- Transicion: δ (00, 00) -> (00, 00, R)

#T2: 00*01$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 00*01
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T3: 0001*1010$001010#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0001*1010
- Estado actual: 00(*10)
- Transicion: δ (00, 10) -> (01, 10, L)

#T4: 00*011010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 00*011010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T5: *00001010$0100#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *00001010
- Estado actual: 01(*00)
- Transicion: δ (01, 00) -> (10, 01, S)

#T6: *01001010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *01001010
- Estado actual: 10(*01)
- No hay transicion aplicable: la maquina se DETIENE 

```
### Prueba N°2

Sumar 1 al numero binario 111: (111 + 1) = (1000)

- M = *010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta = q0*010101
- Cinta final esperada = qf*01000000
- Resultado final = *0100000010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100

#### Salidas del código:

```
#T1: *010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *010101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T2: 01*0101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01*0101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T3: 0101*01$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0101*01
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T4: 010101*10$0010#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 010101*10
- Estado actual: 00(*10)
- Transicion: δ (00, 10) -> (01, 10, L)

#T5: 0101*0110$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0101*0110
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T6: 01*010010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01*010010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T7: *01000010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *01000010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T8: *1000000010$0110#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *1000000010
- Estado actual: 01(*10)
- Transicion: δ (01, 10) -> (10, 01, S)

#T9: *0100000010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *0100000010
- Estado actual: 10(*01)
- No hay transicion aplicable: la maquina se DETIENE 

```

### Prueba N°3

Sumar 1 al numero binario 101111: (101111 + 1) = (110000)

- M = *010001010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta = q0*010001010101
- Cinta final esperada = qf*010100000000
- Resultado final = 01*010000000010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100

#### Salidas del código:

```
#T1: *010001010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: *010001010101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T2: 01*0001010101$0000#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01*0001010101
- Estado actual: 00(*00)
- Transicion: δ (00, 00) -> (00, 00, R)

#T3: 0100*01010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0100*01010101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T4: 010001*010101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 010001*010101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T5: 01000101*0101$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01000101*0101
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T6: 0100010101*01$0001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0100010101*01
- Estado actual: 00(*01)
- Transicion: δ (00, 01) -> (00, 01, R)

#T7: 010001010101*10$0010#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 010001010101*10
- Estado actual: 00(*10)
- Transicion: δ (00, 10) -> (01, 10, L)

#T8: 0100010101*0110$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0100010101*0110
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T9: 01000101*010010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01000101*010010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T10: 010001*01000010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 010001*01000010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T11: 0100*0100000010$0101#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 0100*0100000010
- Estado actual: 01(*01)
- Transicion: δ (01, 01) -> (01, 00, L)

#T12: 01*000000000010$0100#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01*000000000010
- Estado actual: 01(*00)
- Transicion: δ (01, 00) -> (10, 01, S)

#T13: 01*010000000010$1001#0000000001#0001000101#0010011010#0100100100#0101010010#0110100100
- Cinta: 01*010000000010
- Estado actual: 10(*01)
- No hay transicion aplicable: la maquina se DETIENE 

```

## Reflexion Final

La **Máquina de Turing Universal (MTU)** representa el salto conceptual fundamental que permitió la transición de las calculadoras mecánicas o específicas a las **computadoras modernas de propósito general**.

Las Máquinas de Turing estándar estaban diseñadas y "cableadas" para resolver una única tarea específica (por ejemplo, duplicar la cantidad de unos o sumar dos números). Turing creo la MTU como una máquina neutra capaz de simular a cualquier otra maquina M junto con su entrada w.

Las computadoras actuales funcionan exactamente bajo este principio: un único hardware físico puede ejecutar un procesador de texto, un reproductor de video o un modelo de inteligencia artificial sin necesidad de modificar los circuitos del equipo

MTU presentó el **modelo conceptual** que inspiro la creación de la computadora moderna tal como la conocemos: un sistema físico capaz de ejecutar distintos softwares.